Neural Network

In [2]:
import sklearn as skl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

df = pd.read_csv('../DATASETS/salario_tratamento.csv', index_col=0)

In [3]:
X = df.drop('income', axis=1)
y = df['income']

Label encoding

In [4]:
categorical_columns = ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country','age.group']
numeric_columns = ['capital.diff', 'hours.per.week']

label_encoder = LabelEncoder()
for column in categorical_columns:
    X[column] = label_encoder.fit_transform(X[column])

y_encoded = label_encoder.fit_transform(y)

Normalização

In [5]:
scaler = StandardScaler()
X[numeric_columns] = scaler.fit_transform(X[numeric_columns])

Criação do Modelo

In [6]:
model = Sequential()
model.add(Dense(128, input_dim=X.shape[1], activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)



Treino do Modelo

In [7]:
cv_scores = []
cv_confusion_matrices = []
cv_precisions = []
cv_recalls = []

for train_index, test_index in stratified_kfold.split(X, y_encoded):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y_encoded[train_index], y_encoded[test_index]

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    y_pred_proba = model.predict(X_test)
    y_pred = (y_pred_proba > 0.5).astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)


    cv_scores.append(accuracy)
    cv_precisions.append(precision)
    cv_recalls.append(recall)
    cv_confusion_matrices.append(cm)

avg_accuracy = sum(cv_scores) / len(cv_scores)
avg_precision = sum(cv_precisions) / len(cv_precisions)
avg_recall = sum(cv_recalls) / len(cv_recalls)
avg_confusion_matrix = sum(cv_confusion_matrices) / len(cv_confusion_matrices)



204/204 [==============================] - 0s 2ms/step


Resultados

In [8]:
print("Average Accuracy:", avg_accuracy)
print("Average Precision:", avg_precision)
print("Average Recall:", avg_recall)
print("Average Confusion Matrix:\n", avg_confusion_matrix)

Average Accuracy: 0.8429413475072158
Average Precision: 0.7133574688472457
Average Recall: 0.5935609090672598
Average Confusion Matrix:
 [[4558.6  385.4]
 [ 637.4  930.8]]
